# 02. 타입 가드 & 제네릭 (in 연산자 · Generic Class · extends 제약)

이 노트북은 **단독으로 실행**될 수 있도록 01번에서 만든 `User`, `Product` 타입과 더미 데이터를 다시 선언합니다.

- 사용 개념: 유니온 타입(`User | Product`), `in` 연산자 타입 가드, 제네릭 클래스(`<T>`), 제네릭 제약(`T extends Identifiable`),
  접근 제어자 `private`, `T | undefined` 반환값의 안전한 처리
- 규칙: `any`는 한 번도 사용하지 않습니다.

## 0단계. 타입과 더미 데이터 다시 선언

01번 노트북과 같은 타입/데이터입니다. (노트북마다 커널이 따로 실행되므로 다시 선언해야 합니다.)

In [ ]:
// 회원 역할은 세 가지 문자열 중 하나만 허용하는 리터럴 유니온 타입
type UserRole = "admin" | "customer" | "guest";

// 회원(User) 데이터의 모양을 정의하는 인터페이스
interface User {
  id: number;         // 회원 고유 번호
  name: string;       // 회원 이름
  email: string;      // 회원 이메일 (Product에는 없는 속성 → 타입 가드에 활용)
  role: UserRole;     // 회원 역할 (위에서 만든 UserRole만 가능)
  isActive: boolean;  // 활성 회원 여부
}

// 상품 카테고리도 정해진 값만 허용하는 리터럴 유니온 타입
type Category = "electronics" | "clothing" | "books";

// 상품(Product) 데이터의 모양을 정의하는 인터페이스
interface Product {
  id: number;            // 상품 고유 번호
  title: string;         // 상품명
  price: number;         // 정가 (원)
  category: Category;    // 상품 카테고리 (Category만 가능)
  stock: number;         // 재고 수량
  discountRate?: number; // 할인율 (0.1 = 10%), 선택 속성이라 없을 수도 있음 → number | undefined
}

// 회원 더미 데이터: User[] 타입을 명시해서 모든 원소가 User 모양인지 컴파일러가 검사
const users: User[] = [
  { id: 1, name: "김관리", email: "admin@shop.com", role: "admin", isActive: true },     // 관리자
  { id: 2, name: "이고객", email: "lee@shop.com", role: "customer", isActive: true },    // 일반 고객
  { id: 3, name: "박고객", email: "park@shop.com", role: "customer", isActive: false },  // 비활성 고객
  { id: 4, name: "최손님", email: "guest@shop.com", role: "guest", isActive: true },     // 비회원(게스트)
];

// 상품 더미 데이터: discountRate가 있는 상품과 없는 상품을 섞어서 준비
const products: Product[] = [
  { id: 101, title: "무선 키보드", price: 50000, category: "electronics", stock: 10, discountRate: 0.1 }, // 10% 할인
  { id: 102, title: "후드 티셔츠", price: 30000, category: "clothing", stock: 5 },                       // 할인 없음
  { id: 103, title: "타입스크립트 입문서", price: 20000, category: "books", stock: 3, discountRate: 0.2 }, // 20% 할인
];

// 재선언이 정상적으로 되었는지 개수만 간단히 출력
console.log(`회원 ${users.length}명, 상품 ${products.length}개 준비 완료`);

회원 4명, 상품 3개 준비 완료


## 1단계. `renderItemSummary` — `in` 연산자로 타입 좁히기

- 매개변수 `item`은 `User | Product` **유니온 타입**이라, 그대로는 두 타입에 공통인 `id`만 사용할 수 있습니다.
- `'email' in item`은 객체에 `email` 속성이 있는지 검사합니다. `email`은 `User`에만 있으므로
  - `if` 블록 안에서는 `item`이 **`User`** 로 좁혀지고 (`name`, `role` 사용 가능)
  - `else` 블록에서는 남은 타입인 **`Product`** 로 좁혀집니다 (`title`, `price` 사용 가능)

In [ ]:
// 회원 또는 상품을 받아서 한 줄 요약 문자열을 반환하는 함수
function renderItemSummary(item: User | Product): string {
  // 'email' 속성은 User에만 있으므로, 있으면 item은 User 타입으로 좁혀진다
  if ("email" in item) {
    return `[회원] ${item.name} (${item.role})`; // User 전용 속성 name, role 사용 가능
  } else {
    // 여기로 왔다면 email이 없으므로 item은 Product 타입으로 좁혀진다
    return `[상품] ${item.title} - ${item.price}원`; // Product 전용 속성 title, price 사용 가능
  }
}

// 회원과 상품을 하나의 배열에 섞어 담는다 (원소 타입은 User | Product)
// 스프레드 문법(...)으로 두 배열을 펼쳐 하나의 새 배열로 합친다
const mixedItems: (User | Product)[] = [...users, ...products];

// 배열을 돌면서 각 원소의 요약 문자열을 출력
for (const item of mixedItems) {
  console.log(renderItemSummary(item)); // 같은 함수가 타입에 따라 다른 형식으로 출력
}

[회원] 김관리 (admin)


[회원] 이고객 (customer)


[회원] 박고객 (customer)


[회원] 최손님 (guest)


[상품] 무선 키보드 - 50000원


[상품] 후드 티셔츠 - 30000원


[상품] 타입스크립트 입문서 - 20000원


## 2단계. `Identifiable` 인터페이스와 제네릭 클래스 `InMemoryStorage<T>`

- `Identifiable`은 "`id: number`를 가진 객체"라는 **최소 조건**입니다.
- `InMemoryStorage<T extends Identifiable>`는 어떤 타입이든 저장할 수 있는 **제네릭 클래스**지만,
  `extends Identifiable` 제약 때문에 **`id`가 있는 타입만** `T`로 쓸 수 있습니다.
  그래서 클래스 내부에서 `item.id`를 안전하게 사용할 수 있습니다.
- `items` 배열은 `private`이라 클래스 밖에서 직접 수정할 수 없고, 반드시 `add` 메서드를 거쳐야 합니다.
- `getById`는 못 찾을 수 있으므로 반환 타입이 `T | undefined`입니다.
- `getAll`은 내부 배열을 그대로 내보내지 않고 **복사본**(`[...this.items]`)을 반환해 외부에서 원본을 망가뜨리지 못하게 합니다.

In [ ]:
// "id 숫자를 가진 객체"라는 공통 조건을 표현하는 인터페이스
interface Identifiable {
  id: number; // 모든 저장 대상이 반드시 가져야 하는 고유 번호
}

// T는 Identifiable을 만족하는(= id: number가 있는) 타입만 허용하는 제네릭 클래스
class InMemoryStorage<T extends Identifiable> {
  // 실제 데이터를 담는 배열. private이라 클래스 외부에서 storage.items로 접근 불가
  private items: T[] = [];

  // 새 항목을 저장소에 추가하는 메서드 (반환값 없음 → void)
  add(item: T): void {
    this.items.push(item); // 내부 배열 끝에 추가
  }

  // id로 항목을 찾는 메서드. 못 찾으면 undefined를 반환하므로 반환 타입은 T | undefined
  getById(id: number): T | undefined {
    // T extends Identifiable 덕분에 item.id를 타입 에러 없이 사용할 수 있다
    return this.items.find((item) => item.id === id);
  }

  // 저장된 모든 항목을 반환하는 메서드
  getAll(): T[] {
    return [...this.items]; // 스프레드 문법으로 복사본을 만들어 반환 (원본 보호)
  }
}

// [에러 예시] id가 없는 타입은 T로 쓸 수 없다 → Type 'string' does not satisfy the constraint 'Identifiable'.
// const wrongStorage = new InMemoryStorage<string>();

console.log("InMemoryStorage 클래스 정의 완료");

InMemoryStorage 클래스 정의 완료


## 3단계. `User` 저장소와 `Product` 저장소 만들기

- 같은 클래스를 `InMemoryStorage<User>`, `InMemoryStorage<Product>`로 각각 생성합니다.
- `userStorage.add(products[0])`처럼 다른 타입을 넣으면 컴파일 에러가 나므로, 저장소마다 타입이 섞이지 않습니다.

In [ ]:
// User 전용 저장소 생성 (T = User)
const userStorage = new InMemoryStorage<User>();
// Product 전용 저장소 생성 (T = Product)
const productStorage = new InMemoryStorage<Product>();

// 더미 회원을 전부 회원 저장소에 추가
for (const user of users) {
  userStorage.add(user); // 매개변수 타입이 User라서 회원만 들어갈 수 있음
}

// 더미 상품을 전부 상품 저장소에 추가
for (const product of products) {
  productStorage.add(product); // 매개변수 타입이 Product라서 상품만 들어갈 수 있음
}

// [에러 예시] 회원 저장소에 상품을 넣으면 컴파일 에러
// userStorage.add(products[0]);

// 전체 조회: getAll()은 복사본 배열을 돌려준다
console.log("전체 회원:", userStorage.getAll().map((user) => user.name));
console.log("전체 상품:", productStorage.getAll().map((product) => product.title));

전체 회원: [ '김관리', '이고객', '박고객', '최손님' ]


전체 상품: [ '무선 키보드', '후드 티셔츠', '타입스크립트 입문서' ]


## 4단계. `getById` 결과(`T | undefined`)를 안전하게 처리하기

- `getById`의 결과는 `undefined`일 수 있어서, 바로 `.name`에 접근하면 strict 모드에서 에러가 납니다.
- `if (found !== undefined)`로 먼저 확인(타입 좁히기)하고, 없을 때는 안내 메시지를 출력합니다.
- 조회한 결과는 1단계의 `renderItemSummary`로 출력해서 두 기능을 함께 사용합니다.

In [ ]:
// 존재하는 id로 회원 조회 → 결과 타입은 User | undefined
const foundUser: User | undefined = userStorage.getById(2);
if (foundUser !== undefined) {
  // 이 블록 안에서 foundUser는 User로 좁혀져서 속성에 안전하게 접근 가능
  console.log("id 2 회원 조회:", renderItemSummary(foundUser));
} else {
  console.log("id 2 회원을 찾을 수 없습니다.");
}

// 존재하는 id로 상품 조회 → 결과 타입은 Product | undefined
const foundProduct: Product | undefined = productStorage.getById(103);
if (foundProduct !== undefined) {
  console.log("id 103 상품 조회:", renderItemSummary(foundProduct));
} else {
  console.log("id 103 상품을 찾을 수 없습니다.");
}

// 존재하지 않는 id로 조회 → undefined가 반환되는 경우도 안전하게 처리
const missingUser: User | undefined = userStorage.getById(999);
if (missingUser !== undefined) {
  console.log("id 999 회원 조회:", renderItemSummary(missingUser));
} else {
  console.log("id 999 회원을 찾을 수 없습니다. (getById가 undefined 반환)");
}

// 옵셔널 체이닝(?.)과 ??로 한 줄 처리하는 방법도 가능
console.log("id 999 회원 이름:", userStorage.getById(999)?.name ?? "없음");

id 2 회원 조회: [회원] 이고객 (customer)


id 103 상품 조회: [상품] 타입스크립트 입문서 - 20000원


id 999 회원을 찾을 수 없습니다. (getById가 undefined 반환)


id 999 회원 이름: 없음


## 5단계. `getAll()`이 복사본을 반환하는지 확인

`getAll()`로 받은 배열에 항목을 추가해도 저장소 내부(`private items`)는 바뀌지 않아야 합니다.

In [ ]:
// getAll()로 받은 배열(복사본)을 가져온다
const copiedUsers: User[] = userStorage.getAll();
// 복사본에 새 회원을 추가해 본다
copiedUsers.push({ id: 5, name: "정임시", email: "temp@shop.com", role: "guest", isActive: false });

// 복사본은 5명, 저장소 원본은 그대로 4명이어야 한다
console.log(`복사본 길이: ${copiedUsers.length}`);
console.log(`저장소 원본 길이: ${userStorage.getAll().length}`);

복사본 길이: 5


저장소 원본 길이: 4
